In [1]:
from utils import *
import wandb
from datetime import datetime

In [2]:
def itertoolsBetter(dataIter):
    while True:
        for batch in dataIter:
            yield batch

In [3]:
# https://github.com/kreasof-ai/sigreg
def sigreg_weak_loss(x, sketch_dim=64):
    """
    Forces Covariance(x) ~ Identity.
    Matches the 2nd Moment (Spherical Cloud).
    """
    N, C = x.size()
    # 1. Sketching (Optional for C=512, but good for consistency)
    if C > sketch_dim:
        S = torch.randn(sketch_dim, C, device=x.device) / (C ** 0.5)
        x = x @ S.T  # [N, sketch_dim]
    else:
        sketch_dim = C

    # 2. Centering & Covariance
    x = x - x.mean(dim=0, keepdim=True)
    cov = (x.T @ x) / (N - 1 + 1e-6)

    # 3. Target Identity
    target = torch.eye(sketch_dim, device=x.device)

    # 4. Off-diagonal suppression + Diagonal maintenance
    return torch.norm(cov - target, p='fro')


class MoCoQueue:
    def __init__(self, dim: int, size: int = 8192):
        self.queue = nn.functional.normalize(torch.randn(size, dim, device=device), dim=-1)
        self.families = torch.full((size,), fill_value=-1, dtype=torch.long)
        self.ptr = 0
        self.size = size

    def get(self):
        return self.queue.clone(), self.families.clone()

    @torch.no_grad()
    def enqueue(self, embeddings: torch.Tensor, families: torch.Tensor):
        n = embeddings.shape[0]
        # Wrap around if needed
        slots = torch.arange(self.ptr, self.ptr + n) % self.size
        self.queue[slots.to(device)] = nn.functional.normalize(embeddings.detach(), dim=-1)
        self.families[slots] = families.detach().cpu()
        self.ptr = (self.ptr + n) % self.size


class EmbeddingLoss(nn.Module):
    def __init__(self, temperature=0.04, alpha=0.1):
        super().__init__()
        self.temperature = temperature
        self.alpha       = alpha

    def infoNCE(self, queries, keys, queryFamilies, keyFamilies):
        """
        queries: [B, D]  — image or text embeddings from current batch
        keys:    [B+Q, D] — current batch + queue embeddings
        queryFamilies: [B]
        keyFamilies:   [B+Q]
        """
        queries = nn.functional.normalize(queries, dim=-1)
        keys    = nn.functional.normalize(keys,    dim=-1)

        logits = (queries @ keys.T) / self.temperature  # [B, B+Q]

        # Positive mask: same family OR diagonal (exact pair)
        familyMatch = (queryFamilies.unsqueeze(1) == keyFamilies.unsqueeze(0))  # [B, B+Q]
        # Diagonal of the batch portion is always the exact positive
        familyMatch[:, :queryFamilies.shape[0]].fill_diagonal_(True)

        targets = familyMatch.float()
        targets = targets / targets.sum(dim=1, keepdim=True)

        logSoftmax = torch.log_softmax(logits, dim=1)
        return -(targets * logSoftmax).sum(dim=1).mean()

    def forward(self, x, y, families=None, queue: MoCoQueue = None):
        B = x.shape[0]
        device = x.device

        if families is not None and queue is not None:
            queueEmb, queueFam = queue.get()
            queueEmb = queueEmb.to(device)
            queueFam = queueFam.to(device)

            # Keys = current batch + queue for both directions
            imageKeys = torch.cat([x, queueEmb], dim=0)
            allFamilies = torch.cat([families, queueFam], dim=0)

            loss1 = self.infoNCE(y, imageKeys, families, allFamilies)
            loss2 = self.infoNCE(x, y, families, families)

            # Enqueue current batch after computing loss
            queue.enqueue(x, families)

        elif families is not None:
            xNorm, yNorm = nn.functional.normalize(x, dim=-1), nn.functional.normalize(y, dim=-1)
            logits = (xNorm @ yNorm.T) / self.temperature
            familyMatch  = (families.unsqueeze(0) == families.unsqueeze(1))
            familyMatch.fill_diagonal_(True)
            targets = familyMatch.float()
            targets = targets / targets.sum(dim=1, keepdim=True)
            logSoftmax = torch.log_softmax(logits, dim=1)
            logSoftmax2 = torch.log_softmax(logits.T, dim=1)
            loss1 = -(targets * logSoftmax).sum(dim=1).mean()
            loss2 = -(targets.T * logSoftmax2).sum(dim=1).mean()

        else:
            xNorm, yNorm = nn.functional.normalize(x, dim=-1), nn.functional.normalize(y, dim=-1)
            logits = (xNorm @ yNorm.T) / self.temperature
            labels = torch.arange(B, device=device)
            loss1 = nn.functional.cross_entropy(logits,   labels)
            loss2 = nn.functional.cross_entropy(logits.T, labels)

        infoLoss = 0.5 * (loss1 + loss2)
        sigLoss  = 0.5 * (sigreg_weak_loss(x) + sigreg_weak_loss(y))
        return {"total": infoLoss + self.alpha * sigLoss, "info": infoLoss, "sig": sigLoss}


class Perplexity(nn.Module):
    def __init__(self, loss):
        super().__init__()
        self.loss = loss

    def forward(self, y1, y2, names, queue):
        log = self.loss(y1, y2, names, queue)["info"]
        return torch.exp(log)
    

def recallAtK(x, y, families=None, k=10, queue: MoCoQueue = None):
    B = x.shape[0]
    device = x.device
    xNorm = nn.functional.normalize(x, dim=-1)
    yNorm = nn.functional.normalize(y, dim=-1)

    if queue is not None:
        queueEmb, queueFam = queue.get()
        queueEmb = nn.functional.normalize(queueEmb.to(device), dim=-1)
        queueFam = queueFam.to(device)
        keys = torch.cat([xNorm, queueEmb], dim=0)
        allFamilies = torch.cat([families, queueFam], dim=0)
    else:
        keys = xNorm
        allFamilies = families

    logits = yNorm @ keys.T  # [B, B+Q]

    if allFamilies is not None:
        positive = (families.unsqueeze(1) == allFamilies.unsqueeze(0))  # [B, B+Q]
    else:
        positive = torch.zeros(B, keys.shape[0], dtype=torch.bool, device=device)
        positive[:, :B].fill_diagonal_(True)

    topK = logits.topk(k, dim=1).indices
    hits = positive.gather(1, topK).any(dim=1).float()
    return hits.mean().item()

In [4]:
def saveExperiment(imageModel, textModel, config, experimentName, start):
    latestPath = os.path.join("checkpoints", "finetune", "latest")
    if not os.path.exists(os.path.join("checkpoints", "finetune", "latest")):
        os.mkdir(latestPath)

    stamp = start.strftime("%Y-%m-%d %H-%M")
    timePath = os.path.join("checkpoints", "finetune", stamp)
    if not os.path.exists(timePath):
        os.mkdir(timePath)

    saveToPath(latestPath, imageModel, textModel, config, experimentName)
    saveToPath(timePath, imageModel, textModel, config, experimentName)


def saveToPath(path, imageModel, textModel, config, experimentName):
    if not os.path.exists(os.path.join(path, experimentName)):
        os.mkdir(os.path.join(path, experimentName))

    torch.save(imageModel.state_dict(), os.path.join(path, experimentName, "image.pt"))
    # textModel.save_pretrained(os.path.join(path, experimentName, "text"))
    torch.save(textModel.state_dict(), os.path.join(path, experimentName, "text.pt"))
    config.save(os.path.join(path, experimentName, "config.json"))

In [5]:
def getTrainableParams(model):
    return [p for p in model.parameters() if p.requires_grad]

def setFrozen(model, frozen: bool):
    for param in model.parameters():
        param.requires_grad = not frozen

def unfreezeTopBlocks(model, n: int):
    """
    Unfreeze the last n transformer blocks. Assumes your model has a .blocks
    or .layers attribute — adjust to match your actual architecture.
    """
    setFrozen(model, True)
    # Try common attribute names — adjust if yours differs
    blocks = getattr(model, "blocks", None) or getattr(model, "layers", None)
    if blocks is None:
        raise AttributeError("Could not find transformer blocks — check model.blocks or model.layers")
    for block in blocks[-n:]:
        for param in block.parameters():
            param.requires_grad = True
    # Always keep head trainable
    for param in model.head.parameters():
        param.requires_grad = True

def makeOptimizer(config, imageModel, textModel):
    return torch.optim.Adam(
        getTrainableParams(imageModel) + getTrainableParams(textModel),
        lr=config.learningRate
    )

In [6]:
def trainModel(config, textModel, imageModel, dataset, experimentName, start, imageConfig):
    optimizer = torch.optim.Adam(list(textModel.head.parameters()) + list(imageModel.head.parameters()), lr=1e-4)

    imageModel.to(device)
    textModel.to(device)

    queue = MoCoQueue(dim=config.sharedDim, size=8192)
    objective = EmbeddingLoss(temperature=0.04, alpha=0.1)
    # objective = ContrastiveLoss2()
    criterion = Perplexity(objective)

    train, test = dataset.split(dataset, batchSize=config.batchSize)

    testIter = itertoolsBetter(test)

    testHistory = []

    total = 0

    run = None

    try:
        for epoch in range(config.epochs):
            progress = 0
            averageTrainLoss = 0
            averageTestLoss = 0
            for images, info, text, _ in train:
                imageModel.train()
                textModel.train()
                optimizer.zero_grad()

                imageOutputs = imageModel(images.to(device))
                textOutputs = textModel(text.to(device))
                loss = objective(imageOutputs, textOutputs, info.to(device), queue=queue)
                trainPerplexity = criterion(imageOutputs, textOutputs, info.to(device), queue=queue)

                trainLoss = loss["total"].detach().item()
                averageTrainLoss = (averageTrainLoss * progress + trainLoss) / (progress + 1)

                loss["total"].backward()
                optimizer.step()

                with torch.no_grad():
                    imageModel.eval()
                    textModel.eval()
                    images1, info1, text1, _ = next(testIter)
                    imageOutputs1 = imageModel(images1.to(device))
                    textOutputs1 = textModel(text1.to(device))
                    loss1 = objective(imageOutputs1, textOutputs1, info1.to(device), queue=queue)
                    testPerplexity = criterion(imageOutputs1, textOutputs1, info1.to(device), queue=queue)

                    testLoss = loss1["total"].detach().item()
                    averageTestLoss = (averageTestLoss * progress + testLoss) / (progress + 1)

                if run is None:
                    run = wandb.init(entity="dylanberndt123-missouri-state-university", project="Briefcase", config=config.serialize())
                
                run.log({"Train Loss": trainLoss, 
                         "Train Perplexity": trainPerplexity.detach().item(),
                         "Train InfoNCE": loss["info"].detach().item(),
                         "Train SigREG": loss["sig"].detach().item(),
                         "Test Loss": testLoss,
                         "Test Perplexity": testPerplexity.detach().item(),
                         "Test InfoNCE": loss1["info"].detach().item(),
                         "Test SigREG": loss1["sig"].detach().item(),
                         "Train Recall@1": recallAtK(imageOutputs, textOutputs, k=1, families=info.to(device), queue=queue),
                         "Test Recall@1": recallAtK(imageOutputs1, textOutputs1, k=1, families=info1.to(device), queue=queue),
                         "Train Recall@5": recallAtK(imageOutputs, textOutputs, k=5, families=info.to(device), queue=queue),
                         "Test Recall@5": recallAtK(imageOutputs1, textOutputs1, k=5, families=info1.to(device), queue=queue),
                         "Train Recall@10": recallAtK(imageOutputs, textOutputs, k=10, families=info.to(device), queue=queue),
                         "Test Recall@10": recallAtK(imageOutputs1, textOutputs1, k=10, families=info1.to(device), queue=queue)
                         }, step=total)

                progress += 1
                total += 1

                progressString = f"\r{epoch + 1} | {progress}/{len(train)} | {(progress / len(train)) * 100:.3f}%"

                print(f"{progressString} |  Train Loss: {averageTrainLoss:.2f} | Test Loss: {averageTestLoss:.2f}",end="")

            print(f"\rEpoch {epoch + 1} | Train Loss: {averageTrainLoss:.2f} | Test Loss: {averageTestLoss:.2f}{' ' * 50}")

            # if (np.array(testHistory) < averageTestLoss).sum() >= 2:
            #     raise KeyboardInterrupt
            testHistory.append(averageTestLoss)

    except KeyboardInterrupt:
        saveExperiment(imageModel, textModel, imageConfig, experimentName, start)
        return imageModel, textModel

    saveExperiment(imageModel, textModel, imageConfig, experimentName, start)
    return imageModel, textModel

In [7]:
queryConfig = Config().load(os.path.join("configs", "querying.json"))

In [8]:
sharedDim = queryConfig.sharedDim

textModelName = "openai/clip-vit-base-patch32"
textModel = CLIPTextEmbedder(textModelName, sharedDim).to(device)

# textModelName = "bert-base-uncased"
# textModel = BERTTextEmbedder(textModelName, sharedDim).to(device)

vit, imageConfig = ViT.load(os.path.join("checkpoints", "pretrain", "latest"))
imageModel = ViTEmbedder(vit, sharedDim).to(device)

dataset = CombinedQueryData(imageConfig.dataset)
dataset.method = "none"

dataset.setTokenizer(textModelName)

experimentName = "ViT" + " " + textModelName.replace("/", "-")

imageModel, textModel = trainModel(queryConfig, textModel, imageModel, dataset, experimentName, datetime.now(), imageConfig)

saveExperiment(imageModel, textModel, imageConfig, experimentName, datetime.now())

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.l


Loading MyFonts images from dataset ====================


Fonts serialized: 720/3870google/fonts/ofl/kumarone/KumarOne-Regular.ttf execution context too long
Fonts serialized: 939/3870google/fonts/ofl/kumaroneoutline/KumarOneOutline-Regular.ttf execution context too long
Fonts serialized: 2449/3870google/fonts/ofl/notocoloremojicompattest/NotoColorEmojiCompatTest-Regular.ttf invalid pixel size
Fonts serialized: 3870/3870

USING GENERATED QUERIES
1248224 9599 1248224
56.44% of fonts have descriptions


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ubuntu/.netrc.
wandb: Currently logged in as: dylanberndt123 (dylanberndt123-missouri-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1 | Train Loss: 8.08 | Test Loss: 7.90                                                  
Epoch 2 | Train Loss: 7.68 | Test Loss: 7.59                                                  
Epoch 3 | Train Loss: 7.49 | Test Loss: 7.45                                                  
Epoch 4 | Train Loss: 7.39 | Test Loss: 7.40                                                  
Epoch 5 | Train Loss: 7.30 | Test Loss: 7.37                                                  
Epoch 6 | Train Loss: 7.23 | Test Loss: 7.34                                                  
Epoch 7 | Train Loss: 7.16 | Test Loss: 7.33                                                  
Epoch 8 | Train Loss: 7.11 | Test Loss: 7.32                                                  
Epoch 9 | Train Loss: 7.07 | Test Loss: 7.31                                                  
Epoch 10 | Train Loss: 7.02 | Test Loss: 7.32                                                  
Epoch 11 | Train Loss: 6.98 | Test Loss: 7.32    

ConnectionResetError: Connection lost